In [8]:
import yfinance as yf
import pandas as pd
import pandas_ta as ta
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import RFE  
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight

## ------------------------------------------------------------------
## STEP 1: Get the data
## ------------------------------------------------------------------
data = yf.download('SPY', start='2015-01-01', end='2025-1-1')

## ------------------------------------------------------------------
## STEP 2: Create Features
## ------------------------------------------------------------------
# If the columns are a MultiIndex, flatten them
if isinstance(data.columns, pd.MultiIndex):
    data.columns = data.columns.get_level_values(0)

# Add technical indicators using pandas_ta
data.ta.ema(length=100, append=True)  # Exponential Moving Average
data.ta.sma(length=25, append=True)   # Simple Moving Average
data.ta.rsi(length=10, append=True)  # Relative Strength Index
data.ta.vwap(high='High', low='Low', close='Close', volume='Volume', append=True)  # Volume Weighted Average Price
data.ta.bbands(append=True)  # Bollinger Bands
data.ta.atr(length=14, append=True)  # Average True Range

# Example: Add 1-day and 3-day lags for RSI
data['RSI_lag_1'] = data['RSI_10'].shift(1)
data['RSI_lag_3'] = data['RSI_10'].shift(3)

# Example: Add 5-day momentum for SMA
data['SMA_25_momentum'] = data['SMA_25'].pct_change(5)

# Example: Price distance from EMA
data['Price_vs_EMA'] = (data['Close'] - data['EMA_100']) / data['EMA_100']

# Example: Add MACD
data.ta.macd(append=True)

# print column names to verify
print("--- Actual Column Names ---")
print(data.columns)
print("----------------------------")


## ------------------------------------------------------------------
## STEP 3: Create Label
## ------------------------------------------------------------------
future_return = data['Close'].pct_change(1).shift(-1)
UPPER_THRESHOLD = 0.01
LOWER_THRESHOLD = -0.01
conditions = [(future_return > UPPER_THRESHOLD), (future_return < LOWER_THRESHOLD)]
choices = [1, -1]
data['Target'] = np.select(conditions, choices, default=0)
data['Target'] = data['Target'].map({-1: 0, 0: 1, 1: 2})


## ------------------------------------------------------------------
## STEP 4: Prepare Data for the Model (Same)
## ------------------------------------------------------------------
data = data.dropna()
feature_list = [
    'EMA_100', 'SMA_25', 'RSI_10', 'VWAP_D', 'ATRr_14', # Original group
    'BBL_5_2.0_2.0', 'BBB_5_2.0_2.0', 'BBU_5_2.0_2.0', 'BBP_5_2.0_2.0', # BBands group
    
    # --- Add your new features here ---
    'RSI_lag_1', 'RSI_lag_3', 'SMA_25_momentum', 'Price_vs_EMA',
    'MACD_12_26_9', 'MACDh_12_26_9', 'MACDs_12_26_9'    
]
X = data[feature_list]
Y = data['Target']

# split the data into training and testing sets (80% train, 20% test)
split_percentage = 0.8
split_index = int(len(data) * split_percentage)
X_train = X[:split_index]
Y_train = Y[:split_index]
X_test = X[split_index:]
Y_test = Y[split_index:]

print("--- Original Class Distribution (Training Set) ---")
print(Y_train.value_counts(normalize=True))


## ------------------------------------------------------------------
## STEP 6: Train and Evaluate the FINAL Model (No RFE)
## ------------------------------------------------------------------
param_grid_xgb = {
    'n_estimators': [100, 300],
    'max_depth': [3, 5],
    'learning_rate': [0.01, 0.1],   # <-- How fast the model learns
    'subsample': [0.7, 1],          # <-- % of data to use for each tree
    'colsample_bytree': [0.7, 1]    # <-- % of features to use for each tree
}

tscv = TimeSeriesSplit(n_splits=3)
# Create the grid search object
grid_search = GridSearchCV(
    estimator=XGBClassifier(random_state=42, 
                            objective='multi:softmax', # <-- Tell it we have 3 classes
                            num_class=3), 
    param_grid=param_grid_xgb,
    cv=tscv,                 
    scoring='f1_macro',   
    n_jobs=-1             
)

# Calculate weights based on the *original* imbalanced Y_train
sample_weights = compute_sample_weight(
    class_weight='balanced',
    y=Y_train  # <-- Use the original Y_train, not resampled
)

# Fit the grid search on the *original* training data
print(f"\nRunning GridSearchCV on all {len(feature_list)} features...")
grid_search.fit(
    X_train,  # <-- Original X_train
    Y_train,  # <-- Original Y_train
    sample_weight=sample_weights # <-- Pass the weights here
)

# See the best settings it found
print(f"Best parameters found: {grid_search.best_params_}")

# The grid_search object is now your best model
# Predict on the *original, non-resampled* test set (X_test)
predictions = grid_search.predict(X_test)

# Now check the report
print("\n--- Final Classification Report ---")
print(classification_report(Y_test, predictions, target_names=['Sell (-1)', 'Hold (0)', 'Buy (1)']))


/tmp/ipykernel_1056/566610951.py:17: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download('SPY', start='2015-01-01', end='2025-1-1')
[*********************100%***********************]  1 of 1 completed


--- Actual Column Names ---
Index(['Close', 'High', 'Low', 'Open', 'Volume', 'EMA_100', 'SMA_25', 'RSI_10',
       'VWAP_D', 'BBL_5_2.0_2.0', 'BBM_5_2.0_2.0', 'BBU_5_2.0_2.0',
       'BBB_5_2.0_2.0', 'BBP_5_2.0_2.0', 'ATRr_14', 'RSI_lag_1', 'RSI_lag_3',
       'SMA_25_momentum', 'Price_vs_EMA', 'MACD_12_26_9', 'MACDh_12_26_9',
       'MACDs_12_26_9'],
      dtype='object', name='Price')
----------------------------
--- Original Class Distribution (Training Set) ---
Target
1    0.743404
2    0.141231
0    0.115365
Name: proportion, dtype: float64

Running GridSearchCV on all 16 features...
Best parameters found: {'colsample_bytree': 1, 'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.7}

--- Final Classification Report ---
              precision    recall  f1-score   support

   Sell (-1)       0.12      0.61      0.19        44
    Hold (0)       0.89      0.48      0.63       384
     Buy (1)       0.20      0.16      0.18        56

    accuracy            